In [77]:
import os
import optuna
import pandas as pd
from optuna.trial import TrialState
from optuna.study import StudyDirection

In [78]:
import os
import pandas as pd
import optuna
from optuna.trial import TrialState

def best_trial(
    db_path: str,
    study_name: str,
    n_trials: int = 50,
    direction: str = 'min'
):
    study = optuna.load_study(
        study_name=study_name,
        storage=f"sqlite:///{db_path}"
    )

    first_n = sorted(study.trials, key=lambda t: t.number)[:n_trials]
    completed = [t for t in first_n if t.state == TrialState.COMPLETE]

    if not completed:
        raise ValueError("No completed trials found in the first N trials.")

    if direction == 'min':
        best = min(completed, key=lambda t: t.value)
    elif direction == 'max':
        best = max(completed, key=lambda t: t.value)
    else:
        raise ValueError("Direction must be 'min' or 'max'")

    return {
        'trial_number': best.number,
        'value': best.value,
        'learning_rate': best.params.get('learning_rate'),
        'weight_decay': best.params.get('weight_decay'),
        'pooling': best.params.get('pooling'),
        'use_numeric': best.params.get('use_numeric')
    }

# Loop and collect results
dbs_path = '/scratch/sas10092/ehr-foundation/models/optuna_dbs/'
dbs = os.listdir(dbs_path)

rows = []
for db in dbs:
    arch = db.split('_')[0]
    if arch == 'big':
        arch = 'big_bird'
    task = db[len(arch)+1:-3]

    result = best_trial(
        db_path=os.path.join(dbs_path, db),
        study_name=arch,
        n_trials=25,
        direction='min'
    )
    row = {
        'arch': arch,
        'task': task,
        **result
    }
    rows.append(row)

# Convert to DataFrame and reorder columns
df = pd.DataFrame(rows)
df = df[['arch', 'task', 'trial_number', 'value', 'learning_rate', 'weight_decay', 'pooling', 'use_numeric']]

# Optionally save to CSV
# df.to_csv("best_trials_summary.csv", index=False)

# Show result
df = df.sort_values(['task'])
df = df[df.arch != 'bert']
df = df.reset_index(drop=True)

In [79]:
df

,arch,task,trial_number,value,learning_rate,weight_decay,pooling,use_numeric
0,roformer,y_icu_readmit_30,15,0.148925,0.000050,0.002093,mean,True
1,longformer,y_icu_readmit_30,17,0.150567,0.000049,0.003189,mean,False
2,roberta,y_icu_readmit_30,23,0.150211,0.000050,0.005219,cls,False
3,modernbert,y_icu_readmit_30,24,0.159898,0.000050,0.004600,cls,False
4,big_bird,y_icu_readmit_30,3,0.149238,0.000043,0.001881,cls,True
5,roberta,y_los_7,11,0.300067,0.000049,0.001014,mean,True
6,roformer,y_los_7,15,0.278900,0.000045,0.001294,mean,True
7,big_bird,y_los_7,11,0.280478,0.000049,0.001114,cls,False
8,modernbert,y_los_7,2,0.285998,0.000036,0.007465,cls,False
9,longformer,y_los_7,9,0.292982,0.000036,0.001630,mean,False


In [55]:
# change this
study = df.iloc[1]
study

arch             big_bird
task              y_los_7
trial_number           25
value            0.280409
learning_rate    0.000049
weight_decay     0.001957
pooling               cls
use_numeric         False
Name: 5, dtype: object

In [56]:
import os